# Exploracion inicial - NASA Exoplanet Archive + Kepler KOI

Este notebook carga los dos CSV descargados desde NASA Exoplanet Archive, muestra su estructura y clasifica sus columnas por tipo conceptual. La idea es separar el dataset crudo de las columnas que realmente serviran para EDA, warehouse y modelado.


In [31]:
from __future__ import annotations

from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.metrics import pairwise_distances

# Paleta y estilo
NAVY = "#1C3257"
TERRA = "#AA4B37"
SAND = "#F4EFE6"
INK = "#1A1A1A"
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_FONT = dict(family="Helvetica, Arial, sans-serif", color=INK, size=13)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)


In [32]:
def clasificar_atributos(
    df: pd.DataFrame,
    hints: dict[str, str] | None = None,
    roles: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Clasifica columnas por tipo conceptual y rol dentro del proyecto.

    hints permite override manual: {'koi_disposition': 'nominal_objetivo'}.
    roles permite marcar si una columna es id, target, feature, fuga, metadato, etc.
    """
    hints = hints or {}
    roles = roles or {}
    filas = []

    for col in df.columns:
        s = df[col]
        n_unique = int(s.nunique(dropna=True))
        n_missing = int(s.isna().sum())
        pct_missing = round(float(s.isna().mean() * 100), 2)
        dtype = str(s.dtype)

        if col in hints:
            tipo = hints[col]
        elif pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            tipo = "nominal"
        elif n_unique == 2:
            tipo = "binario"
        elif pd.api.types.is_integer_dtype(s) and n_unique <= 12:
            tipo = "ordinal_o_codificado"
        elif pd.api.types.is_numeric_dtype(s):
            tipo = "numerico_continuo"
        else:
            tipo = "revisar"

        ejemplos = s.dropna().astype(str).unique()[:4]
        filas.append(
            {
                "columna": col,
                "dtype_pandas": dtype,
                "tipo_conceptual": tipo,
                "rol_proyecto": roles.get(col, "feature_o_descriptiva"),
                "n_unique": n_unique,
                "n_missing": n_missing,
                "pct_missing": pct_missing,
                "ejemplo_valores": list(ejemplos),
            }
        )

    return pd.DataFrame(filas).sort_values(
        by=["rol_proyecto", "pct_missing", "columna"],
        ascending=[True, False, True],
    ).reset_index(drop=True)


def resumen_dataframe(nombre: str, df: pd.DataFrame) -> dict:
    return {
        "dataset": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "celdas_nulas": int(df.isna().sum().sum()),
        "pct_nulos_total": round(float(df.isna().sum().sum() / df.size * 100), 2),
        "columnas_con_nulos": int((df.isna().sum() > 0).sum()),
    }


def columnas_con_mas_nulos(df: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    missing = df.isna().agg(["sum", "mean"]).T
    missing.columns = ["n_missing", "pct_missing"]
    missing["pct_missing"] = (missing["pct_missing"] * 100).round(2)
    return missing.sort_values("pct_missing", ascending=False).head(n)


def detectar_outliers_iqr(s: pd.Series, k: float = 1.5) -> pd.Series:
    """Mascara booleana: True si el valor es outlier segun la regla k*IQR."""
    if not pd.api.types.is_numeric_dtype(s):
        raise ValueError(f"detectar_outliers_iqr requiere serie numerica, recibio {s.dtype}")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return ((s < low) | (s > high)).fillna(False)


def chi_square(df: pd.DataFrame, col_a: str, col_b: str) -> dict:
    """Chi-cuadrado de independencia entre dos columnas categoricas."""
    for col in (col_a, col_b):
        if col not in df.columns:
            raise ValueError(f"Columna '{col}' no esta en el DataFrame")
    sub = df[[col_a, col_b]].dropna()
    if sub.empty:
        raise ValueError(f"No hay filas con valores en ambas columnas {col_a}, {col_b}")
    observed = pd.crosstab(sub[col_a], sub[col_b])
    chi2, p, dof, expected = stats.chi2_contingency(observed.values)
    expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
    return {
        "chi2": float(chi2),
        "p_value": float(p),
        "dof": int(dof),
        "observed": observed,
        "expected": expected_df,
    }


def aplicar_normalizaciones(s: pd.Series) -> pd.DataFrame:
    """Devuelve original, minmax, zscore y decimal scaling para una serie numerica."""
    if not pd.api.types.is_numeric_dtype(s):
        raise ValueError("aplicar_normalizaciones requiere una Serie numerica")
    x = s.astype(float).reset_index(drop=True)
    rango = x.max() - x.min()
    minmax = (x - x.min()) / rango if rango != 0 else pd.Series(np.zeros(len(x)))
    sigma = x.std()
    zscore = (x - x.mean()) / sigma if sigma != 0 else pd.Series(np.zeros(len(x)))
    max_abs = x.abs().max()
    j = math.ceil(math.log10(max_abs)) if max_abs > 0 else 0
    decimal = x / (10 ** j) if j > 0 else x.copy()
    return pd.DataFrame({"original": x, "minmax": minmax, "zscore": zscore, "decimal": decimal})


def distancia_matriz(df: pd.DataFrame, metrica: str = "euclidean") -> np.ndarray:
    """Matriz de distancia n x n entre filas numericas."""
    permitidas = {"euclidean", "manhattan", "chebyshev", "cosine"}
    if metrica not in permitidas:
        raise ValueError(f"metrica debe ser una de {permitidas}, recibio '{metrica}'")
    no_numericas = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if no_numericas:
        raise ValueError(f"distancia_matriz requiere columnas numericas; no numericas: {no_numericas}")
    return pairwise_distances(df.values, metric=metrica)


## 1. Carga de datos

Los archivos de NASA incluyen lineas iniciales con `#`. Por eso se leen con `comment="#"` para que pandas ignore la metadata y tome como encabezado la primera linea real de columnas.


In [33]:
CANDIDATE_DATA_DIRS = [Path("data"), Path("mineria") / "data"]
DATA_DIR = next(
    (
        data_dir
        for data_dir in CANDIDATE_DATA_DIRS
        if (data_dir / "cumulative_2026.06.01_20.09.17.csv").exists()
        and (data_dir / "PSCompPars_2026.06.01_20.09.10.csv").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontraron los CSV. Deben estar en data/ o mineria/data/."
    )

KEPLER_PATH = DATA_DIR / "cumulative_2026.06.01_20.09.17.csv"
PSCOMP_PATH = DATA_DIR / "PSCompPars_2026.06.01_20.09.10.csv"

kepler = pd.read_csv(KEPLER_PATH, comment="#")
pscomppars = pd.read_csv(PSCOMP_PATH, comment="#")
print(f"Datos cargados desde: {DATA_DIR}")

resumen = pd.DataFrame([
    resumen_dataframe("Kepler KOI cumulative", kepler),
    resumen_dataframe("PSCompPars", pscomppars),
])
resumen


Datos cargados desde: data


,dataset,filas,columnas,celdas_nulas,pct_nulos_total,columnas_con_nulos
0,Kepler KOI cumulative,9564,49,40104,8.56,36
1,PSCompPars,6291,84,78176,14.79,71


In [34]:
print("Kepler KOI - primeras filas")
display(kepler.head(5))

print("PSCompPars - primeras filas")
display(pscomppars.head(5))


Kepler KOI - primeras filas


,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_prad,koi_prad_err1,koi_prad_err2,koi_teq,koi_teq_err1,koi_teq_err2,koi_insol,koi_insol_err1,koi_insol_err2,koi_model_snr,koi_tce_plnt_num,koi_tce_delivname,koi_steff,koi_steff_err1,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,9.488036,2.775000e-05,-2.775000e-05,170.538750,0.002160,-0.002160,0.146,0.318,-0.146,2.95750,0.08190,-0.08190,615.8,19.5,-19.5,2.26,0.26,-0.15,793.0,NaN,NaN,93.59,29.45,-16.65,35.8,1.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,0,54.418383,2.479000e-04,-2.479000e-04,162.513840,0.003520,-0.003520,0.586,0.059,-0.443,4.50700,0.11600,-0.11600,874.8,35.5,-35.5,2.83,0.32,-0.19,443.0,NaN,NaN,9.11,2.87,-1.62,25.8,2.0,q1_q17_dr25_tce,5455.0,81.0,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,10811496,K00753.01,NaN,CANDIDATE,CANDIDATE,0.000,0,0,0,0,19.899140,1.494000e-05,-1.494000e-05,175.850252,0.000581,-0.000581,0.969,5.126,-0.077,1.78220,0.03410,-0.03410,10829.0,171.0,-171.0,14.60,3.92,-1.31,638.0,NaN,NaN,39.30,31.04,-10.49,76.3,1.0,q1_q17_dr25_tce,5853.0,158.0,-176.0,4.544,0.044,-0.176,0.868,0.233,-0.078,297.00482,48.134129,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,0,1.736952,2.630000e-07,-2.630000e-07,170.307565,0.000115,-0.000115,1.276,0.115,-0.092,2.40641,0.00537,-0.00537,8079.2,12.8,-12.8,33.46,8.50,-2.83,1395.0,NaN,NaN,891.96,668.95,-230.35,505.6,1.0,q1_q17_dr25_tce,5805.0,157.0,-174.0,4.564,0.053,-0.168,0.791,0.201,-0.067,285.53461,48.285210,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,0,2.525592,3.761000e-06,-3.761000e-06,171.595550,0.001130,-0.001130,0.701,0.235,-0.478,1.65450,0.04200,-0.04200,603.3,16.9,-16.9,2.75,0.88,-0.35,1406.0,NaN,NaN,926.16,874.33,-314.24,40.9,1.0,q1_q17_dr25_tce,6031.0,169.0,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509


PSCompPars - primeras filas


,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,pl_controv_flag,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxlim,pl_rade,pl_radeerr1,pl_radeerr2,pl_radelim,pl_radj,pl_radjerr1,pl_radjerr2,pl_radjlim,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_bmasselim,pl_bmassj,pl_bmassjerr1,pl_bmassjerr2,pl_bmassjlim,pl_bmassprov,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccenlim,pl_insol,pl_insolerr1,pl_insolerr2,pl_insollim,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtlim,ttv_flag,st_spectype,st_teff,st_tefferr1,st_tefferr2,st_tefflim,st_rad,st_raderr1,st_raderr2,st_radlim,st_mass,st_masserr1,st_masserr2,st_masslim,st_met,st_meterr1,st_meterr2,st_metlim,st_metratio,st_logg,st_loggerr1,st_loggerr2,st_logglim,rastr,ra,decstr,dec,sy_dist,sy_disterr1,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2
0,11 Com b,11 Com,2,1,Radial Velocity,2007.0,Xinglong Station,0,323.21000,0.06,-0.05,0.0,1.178,0.000,0.000,0.0,12.2,NaN,NaN,0.0,1.09,NaN,NaN,0.0,4914.898486,39.092894,-39.728551,0.0,15.464,0.123,-0.125,0.0,Msini,0.2380,0.0070,-0.0070,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,G8 III,4874.0,NaN,NaN,0.0,13.76,2.85,-2.45,0.0,2.09,0.64,-0.63,0.0,-0.26,0.10,-0.10,0.0,[Fe/H],2.45,0.08,-0.08,0.0,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,93.1846,1.9238,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848
1,11 UMi b,11 UMi,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,0,516.21997,3.20,-3.20,0.0,1.530,0.070,-0.070,0.0,12.3,NaN,NaN,0.0,1.09,NaN,NaN,0.0,4684.814200,794.575000,-794.575000,0.0,14.740,2.500,-2.500,0.0,Msini,0.0800,0.0300,-0.0300,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,K4 III,4213.0,46.0,-46.0,0.0,29.79,2.84,-2.84,0.0,2.78,0.69,-0.69,0.0,-0.02,NaN,NaN,0.0,[Fe/H],1.93,0.07,-0.07,0.0,15h17m05.90s,229.274595,+71d49m26.19s,71.823943,125.3210,1.9765,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903
2,14 And b,14 And,1,1,Radial Velocity,2008.0,Okayama Astrophysical Observatory,0,186.76000,0.11,-0.12,0.0,0.775,0.000,0.000,0.0,13.1,NaN,NaN,0.0,1.16,NaN,NaN,0.0,1131.151301,36.232438,-38.775066,0.0,3.559,0.114,-0.122,0.0,Msini,0.0000,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,K0 III,4888.0,NaN,NaN,0.0,11.55,1.12,-0.51,0.0,1.78,0.43,-0.29,0.0,-0.21,0.10,-0.10,0.0,[Fe/H],2.55,0.06,-0.07,0.0,23h31m17.80s,352.824150,+39d14m09.01s,39.235837,75.4392,0.7140,-0.7140,5.23133,0.023,-0.023,2.331,0.240,-0.240,4.91781,0.002826,-0.002826
3,14 Her b,14 Her,1,2,Radial Velocity,2002.0,W. M. Keck Observatory,0,1766.41000,0.67,-0.68,0.0,2.839,0.039,-0.041,0.0,12.5,NaN,NaN,0.0,1.12,NaN,NaN,0.0,2828.672822,413.176929,-540.308292,0.0,8.900,1.300,-1.700,0.0,Mass,0.3683,0.0029,-0.0029,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,K0,5338.0,25.0,-25.0,0.0,0.93,0.01,-0.01,0.0,0.97,0.04,-0.04,0.0,0.43,0.07,-0.07,0.0,[Fe/H],4.45,0.02,-0.02,0.0,16h10m24.50s,242.602101,+43d48m58.90s,43.816362,17.9323,0.0073,-0.0073,6.61935,0.023,-0.023,4.714,0.016,-0.016,6.38300,0.000351,-0.000351
4,16 Cyg B b,16 Cyg B,3,1,Radial Velocity,1996.0,Multiple Observatories,0,798.50000,1.00,-1.00,0.0,1.660,0.030,-0.030,0.0,13.5,NaN,NaN,0.0,1.20,NaN,NaN,0.0,565.737400,25.426400,-25.426400,0.0,1.780,0.080,-0.080,0.0,Msini,0.6800,0.0200,-0.0200,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,G3 V,5750.0,8.0,-8.0,0.0,1.13,0.01,-0.01,0.0,1.08,0.04,-0.04,0.0,0.06,NaN,NaN,0.0,[Fe/H],4.36,0.01,-0.01,0.0,19h41m51.75s,295.465642,+50d31m00.57s,50.516824,21.1397,0.0110,-0.0111,6.21500,0.016,-0.016,4.651,0.016,-0.016,6.06428,0.000603,-0.000603


## 2. Columnas y valores faltantes

Esta revision ayuda a decidir que columnas entran al EDA, al warehouse y a los modelos. No todas las columnas del archivo crudo deben usarse para entrenar.


In [35]:
print("Columnas Kepler KOI")
display(pd.DataFrame({"columna": kepler.columns}))

print("Columnas PSCompPars")
display(pd.DataFrame({"columna": pscomppars.columns}))


Columnas Kepler KOI


,columna
0,kepid
1,kepoi_name
2,kepler_name
3,koi_disposition
4,koi_pdisposition
5,koi_score
6,koi_fpflag_nt
7,koi_fpflag_ss
8,koi_fpflag_co
9,koi_fpflag_ec


Columnas PSCompPars


,columna
0,pl_name
1,hostname
2,sy_snum
3,sy_pnum
4,discoverymethod
...,...
79,sy_kmagerr1
80,sy_kmagerr2
81,sy_gaiamag
82,sy_gaiamagerr1


In [36]:
print("Kepler KOI - columnas con mas nulos")
display(columnas_con_mas_nulos(kepler))

print("PSCompPars - columnas con mas nulos")
display(columnas_con_mas_nulos(pscomppars))


Kepler KOI - columnas con mas nulos


,n_missing,pct_missing
koi_teq_err1,9564.0,100.00
koi_teq_err2,9564.0,100.00
kepler_name,6817.0,71.28
koi_score,1510.0,15.79
koi_steff_err2,483.0,5.05
koi_srad_err1,468.0,4.89
koi_steff_err1,468.0,4.89
koi_slogg_err2,468.0,4.89
koi_slogg_err1,468.0,4.89
koi_srad_err2,468.0,4.89


PSCompPars - columnas con mas nulos


,n_missing,pct_missing
pl_eqterr1,4460.0,70.89
pl_eqterr2,4460.0,70.89
pl_orbeccenerr1,4411.0,70.12
pl_orbeccenerr2,4411.0,70.12
st_spectype,3967.0,63.06
pl_bmasseerr1,3274.0,52.04
pl_bmassjerr2,3274.0,52.04
pl_bmasseerr2,3274.0,52.04
pl_bmassjerr1,3274.0,52.04
pl_insolerr2,2645.0,42.04


## 3. Clasificacion conceptual de atributos

La clasificacion separa identificadores, objetivos de modelado, variables numericas, nominales, metadatos y columnas que pueden causar fuga de datos.


In [37]:
KEPLER_HINTS = {
    "kepid": "identificador",
    "kepoi_name": "identificador",
    "kepler_name": "identificador",
    "koi_disposition": "nominal_objetivo",
    "koi_pdisposition": "nominal_con_fuga",
    "koi_score": "numerico_con_fuga",
    "koi_fpflag_nt": "binario_con_fuga",
    "koi_fpflag_ss": "binario_con_fuga",
    "koi_fpflag_co": "binario_con_fuga",
    "koi_fpflag_ec": "binario_con_fuga",
    "koi_period": "numerico_continuo",
    "koi_impact": "numerico_continuo",
    "koi_duration": "numerico_continuo",
    "koi_depth": "numerico_continuo",
    "koi_prad": "numerico_objetivo_regresion",
    "koi_teq": "numerico_continuo",
    "koi_insol": "numerico_continuo",
    "koi_model_snr": "numerico_continuo",
    "koi_steff": "numerico_continuo",
    "koi_slogg": "numerico_continuo",
    "koi_srad": "numerico_continuo",
    "ra": "coordenada",
    "dec": "coordenada",
    "koi_kepmag": "numerico_continuo",
}

KEPLER_ROLES = {
    "kepid": "id",
    "kepoi_name": "id",
    "kepler_name": "id",
    "koi_disposition": "target_clasificacion",
    "koi_prad": "target_regresion_posible",
    "koi_pdisposition": "no_usar_fuga",
    "koi_score": "no_usar_fuga",
    "koi_fpflag_nt": "no_usar_fuga",
    "koi_fpflag_ss": "no_usar_fuga",
    "koi_fpflag_co": "no_usar_fuga",
    "koi_fpflag_ec": "no_usar_fuga",
    "koi_tce_delivname": "metadato",
    "ra": "dimension_espacial",
    "dec": "dimension_espacial",
}

clasificacion_kepler = clasificar_atributos(kepler, hints=KEPLER_HINTS, roles=KEPLER_ROLES)
clasificacion_kepler


,columna,dtype_pandas,tipo_conceptual,rol_proyecto,n_unique,n_missing,pct_missing,ejemplo_valores
0,dec,float64,coordenada,dimension_espacial,8195,0,0.00,"[48.141651, 48.134129, 48.28521, 48.2262]"
1,ra,float64,coordenada,dimension_espacial,8131,0,0.00,"[291.93423, 297.00482, 285.53461, 288.75488]"
2,koi_teq_err1,float64,numerico_continuo,feature_o_descriptiva,0,9564,100.00,[]
3,koi_teq_err2,float64,numerico_continuo,feature_o_descriptiva,0,9564,100.00,[]
4,koi_steff_err2,float64,numerico_continuo,feature_o_descriptiva,376,483,5.05,"[-81.0, -176.0, -174.0, -211.0]"
5,koi_slogg_err1,float64,numerico_continuo,feature_o_descriptiva,553,468,4.89,"[0.064, 0.044, 0.053, 0.07]"
6,koi_slogg_err2,float64,numerico_continuo,feature_o_descriptiva,430,468,4.89,"[-0.096, -0.176, -0.168, -0.21]"
7,koi_srad_err1,float64,numerico_continuo,feature_o_descriptiva,1162,468,4.89,"[0.105, 0.233, 0.201, 0.334]"
8,koi_srad_err2,float64,numerico_continuo,feature_o_descriptiva,1384,468,4.89,"[-0.061, -0.078, -0.067, -0.133]"
9,koi_steff_err1,float64,numerico_continuo,feature_o_descriptiva,265,468,4.89,"[81.0, 158.0, 157.0, 169.0]"


In [38]:
PSCOMP_HINTS = {
    "pl_name": "identificador",
    "hostname": "identificador",
    "discoverymethod": "nominal",
    "disc_facility": "nominal",
    "disc_year": "ordinal_temporal",
    "pl_orbper": "numerico_continuo",
    "pl_orbsmax": "numerico_continuo",
    "pl_rade": "numerico_continuo",
    "pl_radj": "numerico_continuo",
    "pl_bmasse": "numerico_continuo",
    "pl_bmassj": "numerico_continuo",
    "pl_eqt": "numerico_continuo",
    "pl_insol": "numerico_continuo",
    "st_teff": "numerico_continuo",
    "st_rad": "numerico_continuo",
    "st_mass": "numerico_continuo",
    "st_met": "numerico_continuo",
    "st_logg": "numerico_continuo",
    "ra": "coordenada",
    "dec": "coordenada",
    "sy_dist": "numerico_continuo",
}

PSCOMP_ROLES = {
    "pl_name": "id",
    "hostname": "id",
    "discoverymethod": "dimension_descubrimiento",
    "disc_facility": "dimension_descubrimiento",
    "disc_year": "dimension_tiempo",
    "pl_rade": "target_regresion_posible",
    "pl_orbper": "feature_o_medida",
    "st_teff": "feature_o_medida",
    "st_rad": "feature_o_medida",
    "st_mass": "feature_o_medida",
    "ra": "dimension_espacial",
    "dec": "dimension_espacial",
}

clasificacion_pscomppars = clasificar_atributos(pscomppars, hints=PSCOMP_HINTS, roles=PSCOMP_ROLES)
clasificacion_pscomppars


,columna,dtype_pandas,tipo_conceptual,rol_proyecto,n_unique,n_missing,pct_missing,ejemplo_valores
0,disc_facility,str,nominal,dimension_descubrimiento,73,0,0.00,"[Xinglong Station, Thueringer Landessternwarte Tautenburg, Okayama Astrophys..."
1,discoverymethod,str,nominal,dimension_descubrimiento,11,0,0.00,"[Radial Velocity, Imaging, Eclipse Timing Variations, Microlensing]"
2,dec,float64,coordenada,dimension_espacial,4706,0,0.00,"[17.7932516, 71.8239428, 39.2358367, 43.8163621]"
3,ra,float64,coordenada,dimension_espacial,4706,0,0.00,"[185.1787793, 229.2745954, 352.82415, 242.6021014]"
4,disc_year,float64,ordinal_temporal,dimension_tiempo,34,1,0.02,"[2007.0, 2009.0, 2008.0, 2002.0]"
...,...,...,...,...,...,...,...,...
79,st_teff,float64,numerico_continuo,feature_o_medida,2416,294,4.67,"[4874.0, 4213.0, 4888.0, 5338.0]"
80,st_mass,float64,numerico_continuo,feature_o_medida,1231,9,0.14,"[2.09, 2.78, 1.78, 0.97]"
81,hostname,str,identificador,id,4709,0,0.00,"[11 Com, 11 UMi, 14 And, 14 Her]"
82,pl_name,str,identificador,id,6291,0,0.00,"[11 Com b, 11 UMi b, 14 And b, 14 Her b]"


## 4. Variables objetivo y columnas recomendadas

Para el primer corte se necesita una tarea de clasificacion y una de regresion. Estas son las columnas candidatas y las columnas que se deben excluir para evitar fuga de datos.


In [39]:
columnas_fuga_kepler = [
    "koi_pdisposition",
    "koi_score",
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec",
]

features_kepler_base = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag",
]

target_clasificacion = "koi_disposition"
target_regresion = "koi_prad"

resumen_modelado = pd.DataFrame(
    {
        "uso": [
            "target_clasificacion",
            "target_regresion",
            "features_base",
            "excluir_por_fuga",
        ],
        "columnas": [
            target_clasificacion,
            target_regresion,
            ", ".join(features_kepler_base),
            ", ".join(columnas_fuga_kepler),
        ],
    }
)
resumen_modelado


,uso,columnas
0,target_clasificacion,koi_disposition
1,target_regresion,koi_prad
2,features_base,"koi_period, koi_impact, koi_duration, koi_depth, koi_teq, koi_insol, koi_mod..."
3,excluir_por_fuga,"koi_pdisposition, koi_score, koi_fpflag_nt, koi_fpflag_ss, koi_fpflag_co, ko..."


In [40]:
print("Distribucion del target de clasificacion: koi_disposition")
display(kepler[target_clasificacion].value_counts(dropna=False).rename_axis("clase").reset_index(name="n"))

print("Resumen del target de regresion: koi_prad")
display(kepler[target_regresion].describe().to_frame().T)

print("Metodos de descubrimiento en PSCompPars")
display(pscomppars["discoverymethod"].value_counts(dropna=False).head(12).rename_axis("metodo").reset_index(name="n"))


Distribucion del target de clasificacion: koi_disposition


,clase,n
0,FALSE POSITIVE,4839
1,CONFIRMED,2747
2,CANDIDATE,1978


Resumen del target de regresion: koi_prad


,count,mean,std,min,25%,50%,75%,max
koi_prad,9201.0,102.891778,3077.639126,0.08,1.4,2.39,14.93,200346.0


Metodos de descubrimiento en PSCompPars


,metodo,n
0,Transit,4651
1,Radial Velocity,1181
2,Microlensing,278
3,Imaging,97
4,Transit Timing Variations,41
5,Eclipse Timing Variations,17
6,Orbital Brightness Modulation,9
7,Pulsar Timing,8
8,Astrometry,6
9,Pulsation Timing Variations,2


## 5. Recorte limpio para EDA y modelado inicial

Este subconjunto conserva las columnas utiles para las primeras pruebas y deja fuera las columnas con fuga. Todavia no entrena modelos; solo prepara la base que despues se usara en el pipeline.


In [41]:
columnas_kepler_modelo = [
    "kepid",
    "kepoi_name",
    "kepler_name",
    target_clasificacion,
    target_regresion,
    *features_kepler_base,
]

kepler_modelo = kepler[columnas_kepler_modelo].copy()
kepler_modelo = kepler_modelo.dropna(subset=[target_clasificacion])

print(f"Dataset Kepler reducido: {kepler_modelo.shape[0]} filas x {kepler_modelo.shape[1]} columnas")
display(kepler_modelo.head(10))

display(columnas_con_mas_nulos(kepler_modelo, n=20))


Dataset Kepler reducido: 9564 filas x 18 columnas


,kepid,kepoi_name,kepler_name,koi_disposition,koi_prad,koi_period,koi_impact,koi_duration,koi_depth,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,ra,dec,koi_kepmag
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,2.26,9.488036,0.146,2.95750,615.8,793.0,93.59,35.8,5455.0,4.467,0.927,291.93423,48.141651,15.347
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,2.83,54.418383,0.586,4.50700,874.8,443.0,9.11,25.8,5455.0,4.467,0.927,291.93423,48.141651,15.347
2,10811496,K00753.01,NaN,CANDIDATE,14.60,19.899140,0.969,1.78220,10829.0,638.0,39.30,76.3,5853.0,4.544,0.868,297.00482,48.134129,15.436
3,10848459,K00754.01,NaN,FALSE POSITIVE,33.46,1.736952,1.276,2.40641,8079.2,1395.0,891.96,505.6,5805.0,4.564,0.791,285.53461,48.285210,15.597
4,10854555,K00755.01,Kepler-664 b,CONFIRMED,2.75,2.525592,0.701,1.65450,603.3,1406.0,926.16,40.9,6031.0,4.438,1.046,288.75488,48.226200,15.509
5,10872983,K00756.01,Kepler-228 d,CONFIRMED,3.90,11.094321,0.538,4.59450,1517.5,835.0,114.81,66.5,6046.0,4.486,0.972,296.28613,48.224670,15.714
6,10872983,K00756.02,Kepler-228 c,CONFIRMED,2.77,4.134435,0.762,3.14020,686.0,1160.0,427.65,40.2,6046.0,4.486,0.972,296.28613,48.224670,15.714
7,10872983,K00756.03,Kepler-228 b,CONFIRMED,1.59,2.566589,0.755,2.42900,226.5,1360.0,807.74,15.0,6046.0,4.486,0.972,296.28613,48.224670,15.714
8,6721123,K00114.01,NaN,FALSE POSITIVE,39.21,7.361790,1.169,5.02200,233.7,1342.0,767.22,47.7,6227.0,3.986,1.958,298.86435,42.151569,12.660
9,10910878,K00757.01,Kepler-229 c,CONFIRMED,5.76,16.068647,0.052,3.53470,4914.3,600.0,30.75,161.9,5031.0,4.485,0.848,286.99948,48.375790,15.841


,n_missing,pct_missing
kepler_name,6817.0,71.28
koi_prad,363.0,3.80
koi_srad,363.0,3.80
koi_slogg,363.0,3.80
koi_depth,363.0,3.80
koi_impact,363.0,3.80
koi_teq,363.0,3.80
koi_model_snr,363.0,3.80
koi_steff,363.0,3.80
koi_insol,321.0,3.36
